In [2]:
import sys

!{sys.executable} -m pip install \
langchain-community \
langchain-text-splitters \
langchain-groq \
langchain-huggingface \
sentence-transformers \
pypdf \
scikit-learn \
tiktoken

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from sklearn.metrics.pairwise import cosine_similarity

from langchain_groq import ChatGroq

import numpy as np

import tiktoken


class PDFChatbot:

    def __init__(self, pdf_path, groq_api_key):

        self.pdf_path = "/Users/Aryaanil/Documents/survey cdc.pdf"

        self.groq_api_key = "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"

        self.history = []   
        self.last_sources = []

        self.token_limit = 3000

        self.tokenizer = tiktoken.get_encoding(
            "cl100k_base"
        )

        self.load_pdf()

        self.create_embeddings()

        self.load_llm()


    def load_pdf(self):

        loader = PyPDFLoader(self.pdf_path)

        documents = loader.load()

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=50
        )

        self.chunks = splitter.split_documents(
            documents
        )

        self.texts = [
            doc.page_content for doc in self.chunks
        ]

        self.metadata = [
            doc.metadata for doc in self.chunks
        ]


    def create_embeddings(self):

        self.embedding_model = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )

        embeddings = self.embedding_model.embed_documents(
            self.texts
        )

        self.document_embeddings = np.array(
            embeddings
        )


    def load_llm(self):

        self.llm = ChatGroq(
            groq_api_key=self.groq_api_key,
            model_name="llama-3.3-70b-versatile"
        )


    def retrieve(self, query, k=3):

        query_embedding = self.embedding_model.embed_query(
            query
        )

        similarity_scores = cosine_similarity(
            [query_embedding],
            self.document_embeddings
        )[0]

        top_indices = similarity_scores.argsort()[::-1][:k]

        retrieved = []

        for idx in top_indices:

            retrieved.append({
                "text": self.texts[idx],
                "metadata": self.metadata[idx]
            })

        self.last_sources = retrieved

        return retrieved


    def count_tokens(self):

        history_text = ""

        for message in self.history:

            history_text += (
                f"{message['role']}: "
                f"{message['content']}\n"
            )

        tokens = self.tokenizer.encode(
            history_text
        )

        return len(tokens)


    def trim_history(self):

        total_tokens = self.count_tokens()

        print(
            f"\nCurrent History Tokens: "
            f"{total_tokens}"
        )

        if total_tokens > self.token_limit:

            self.history = self.history[-8:]

            print(
                "\nHistory exceeded 3000 tokens."
            )

            print(
                "History trimmed to last 4 exchanges."
            )


    def conversation_summary(self):

        history_text = ""

        for message in self.history:

            history_text += (
                f"{message['role']}: "
                f"{message['content']}\n"
            )

        prompt = f"""
        Summarize this conversation
        in exactly 3 bullet points.

        Conversation:
        {history_text}
        """

        response = self.llm.invoke(prompt)

        return response.content


    def show_sources(self):

        print("\nLAST RETRIEVED SOURCES")

        for i, source in enumerate(
            self.last_sources,
            start=1
        ):

            page = source["metadata"].get(
                "page",
                "Unknown"
            )

            preview = source["text"][:100].replace(
                "\n",
                " "
            )

            print(f"\nSource {i}")

            print(f"Page: {page}")

            print(f"Preview: {preview}...")


    def ask(self, question):

        if question.lower() == "summary":

            summary = self.conversation_summary()

            print("\nSUMMARY")

            print(summary)

            return summary


        if question.lower() == "sources":

            self.show_sources()

            return "Sources displayed."


        retrieved_docs = self.retrieve(question)

        context = "\n".join(
            [doc["text"] for doc in retrieved_docs]
        )

        history_text = ""

        for message in self.history:

            history_text += (
                f"{message['role']}: "
                f"{message['content']}\n"
            )


        prompt = f"""
        You are a helpful PDF chatbot.

        Use conversation history and
        document context to answer.

        Conversation History:
        {history_text}

        Context:
        {context}

        User Question:
        {question}
        """

        response = self.llm.invoke(prompt)

        answer = response.content


        self.history.append({
            "role": "user",
            "content": question
        })

        self.history.append({
            "role": "assistant",
            "content": answer
        })


        self.trim_history()

        return answer


pdf_path = "/Users/Aryaanil/Documents/survey cdc.pdf"

groq_api_key = "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"

chatbot = PDFChatbot(
    pdf_path,
    groq_api_key
)


conversation = [

    "What is the main topic of the PDF?",

    "Explain the topic simply.",

    "summary",

    "Which libraries are mentioned?",

    "sources",

    "How are those libraries used?"
]


for turn, question in enumerate(
    conversation,
    start=1
):

    print(f"\nTURN {turn}")

    print("\nUSER:")
    print(question)

    response = chatbot.ask(question)

    if question.lower() not in [
        "summary",
        "sources"
    ]:

        print("\nBOT:")
        print(response)

    print("\n" + "=" * 70)


print("\nFINAL MESSAGE HISTORY")


for message in chatbot.history:

    print(f"\n{message['role'].upper()}:")

    print(message["content"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


TURN 1

USER:
What is the main topic of the PDF?

Current History Tokens: 46

BOT:
The main topic of the PDF appears to be related to academic and student-related matters, with a focus on the student community, specifically regarding stress and handling difficult situations.


TURN 2

USER:
Explain the topic simply.

Current History Tokens: 120

BOT:
The topic of the PDF is about students and how they feel, especially when it comes to stress and being happy. It's based on a survey that asked students questions to understand their experiences and perspectives. One of the questions is about how happy they feel in their daily life, on a scale of 1 to 10.


TURN 3

USER:
summary

SUMMARY
* The main topic of the PDF is related to academic and student-related matters, focusing on the student community, specifically regarding stress and handling difficult situations.
* The topic can be simplified to understanding how students feel, particularly in relation to stress and happiness, based on a